In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
from transformers import Qwen2VLForConditionalGeneration, Qwen2VLProcessor
import torch

In [ ]:
model = Qwen2VLForConditionalGeneration.from_pretrained(
    "MinhQuy24/Qwen-2B-ViVQA",
    device_map="auto",
    load_in_4bit=True
)
processor = Qwen2VLProcessor.from_pretrained("MinhQuy24/Qwen-2B-ViVQA")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


adapter_config.json: 0.00B [00:00, ?B/s]

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:239: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/392 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/932 [00:00<?, ?B/s]

In [ ]:
def get_answer(image_path, question, model, processor, max_new_tokens=48):
    # Tạo prompt đúng định dạng multimodal
    messages = [
        {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": question}]}
    ]
    prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(
        images=[image_path],
        text=[prompt],
        return_tensors="pt"
    ).to(model.device)
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
        )
    generated_ids = output[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

In [ ]:
image_path = "/content/tải xuống.jpg"
question = "Mô tả thật chi tiết bức ảnh này cho tôi "
answer = get_answer(image_path, question,model,processor)
answer

'một con mèo đang nằm trên một cái ghế'

In [ ]:
!pip install fastapi uvicorn python-multipart nest-asyncio gradio requests

In [ ]:
import io
import threading
import nest_asyncio
import uvicorn
import gradio as gr
import requests
from fastapi import FastAPI, UploadFile, File, Form
from PIL import Image
from fastapi.middleware.cors import CORSMiddleware

# Allow Uvicorn to run in Jupyter/Colab environment
nest_asyncio.apply()

app = FastAPI(title="Vi-VQA API Service")

# Configure CORS
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# --- BACK-END SECTION ---

@app.post("/predict")
async def predict_api(image: UploadFile = File(...), question: str = Form(...)):
    image_data = await image.read()
    pil_image = Image.open(io.BytesIO(image_data)).convert("RGB")

    try:
        answer = get_answer(pil_image, question, model, processor)
        return {"answer": answer}
    except Exception as e:
        return {"error": str(e)}

def run_api():
    uvicorn.run(app, host="127.0.0.1", port=8000)

# Run Backend in background thread
backend_thread = threading.Thread(target=run_api, daemon=True)
backend_thread.start()

# --- FRONT-END SECTION ---

def vqa_ui_wrapper(image_file, user_question):
    if image_file is None or not user_question:
        return "Please provide both an image and a question."

    url = "http://127.0.0.1:8000/predict"
    with open(image_file, "rb") as f:
        files = {"image": f}
        data = {"question": user_question}
        response = requests.post(url, files=files, data=data)

    if response.status_code == 200:
        res_json = response.json()
        return res_json.get("answer", res_json.get("error", "Unknown Error"))
    else:
        return f"Server Error: {response.status_code}"

# Create Gradio UI in English
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("## 📷 Vietnamese Visual Question Answering (Vi-VQA)")
    gr.Markdown("Research project on knowledge distillation from Qwen2-VL-7B to 2B.")

    with gr.Row():
        with gr.Column():
            input_img = gr.Image(type='filepath', label='Upload Image')
            input_txt = gr.Textbox(lines=2, label='Ask a Question', placeholder='Example: What is the person doing in the photo?')
            btn = gr.Button('Submit Request', variant='primary')

        with gr.Column():
            output_txt = gr.Textbox(label='AI Response', interactive=False)

    btn.click(fn=vqa_ui_wrapper, inputs=[input_img, input_txt], outputs=output_txt)

# Launch UI with English instructions
demo.launch(share=True, debug=True)

/tmp/ipykernel_3707/1707693193.py:63: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:
INFO:     Started server process [3707]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('127.0.0.1', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://6d59d086993c849ff2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
